# ArchaeoTrace: USGS + Korean AMS weak-label contour experiment

This notebook retrains the contour guide on **machine-generated weak labels** from two map families: the 144 USGS crops of the first experiment and 149 crops from 16 black-ink AMS Korea 1:50,000 (Series L751) sheets. No label is human reviewed.

Korean labels come from stroke geometry on black print. Long, thin, curved, unbranched strokes are contour candidates; blue overprint, straight grid lines, compact glyphs and filled symbols are non-contour ink; everything unresolved is excluded from the loss.

Two training augmentations address what the first model got wrong:

- `ink_recolor` repaints labelled contour ink black, grey, blue or brown, so colour alone cannot identify a contour.
- `label_gaps` writes elevation-style numbers across labelled contours and cuts the printed line, while the contour target stays continuous. The model learns to treat the number as non-contour ink and to join the cut contour.

Seeds 3407, 3408 and 3409 start from ImageNet ResNet18 and select by weak-label validation loss only. Calibration and holdout crops stay unopened. Validation loss measures agreement with weak labels; it **does not establish contour accuracy, formal holdout performance or general Korean-map accuracy**.

In [ ]:
from pathlib import Path

REPOSITORY = "https://github.com/lzpxilfe/AI-Vectorizer-for-Archaeology.git"
CODE_COMMIT = "ea3a2d1a4525b0f0cd10c0746882d429686f1c4b"
DATASET_ZIP = "/content/drive/MyDrive/ArchaeoTrace/ai-weak-korea/weak-dataset.zip"
DATASET_PARTS_MANIFEST = "/content/drive/MyDrive/ArchaeoTrace/ai-weak-korea/weak-dataset.parts.json"
MANIFEST_RELATIVE_PATH = "manifest.json"
OUTPUT_DIRECTORY = "/content/drive/MyDrive/ArchaeoTrace/ai-weak-korea/runs"
SEEDS = (3407, 3408, 3409)
RESUME_EXISTING = True
OOM_RETRY = False  # Deliberate retry: resume latest.pt at batch 1 / accumulation 16.
MODEL_LICENSE = "GPL-2.0-only"
ALLOW_WEAK_LABELS = True  # Explicit experimental opt-in; never human approval.
DATASET_ID = "usgs-ams-korea-contour-ai-weak-v1"
EXPECTED_SOURCE_COUNTS = {"train": 14, "validation": 4, "calibration": 6, "holdout": 8}
EXPECTED_SAMPLE_COUNTS = {"train": 162, "validation": 48, "calibration": 36, "holdout": 47}
TRAIN_OPTIONS = {"ink_recolor": True, "label_gaps": True}
MODEL_ID = "ai-weak-contour-guide-korea-v1"


Connect the existing GPU and mount your own Drive. No GPU subscription or compute-unit purchase is performed. Drive authorization is handled by Colab; credentials are never written into the dataset or model bundle.


In [ ]:
import re, subprocess, sys
from google.colab import drive

if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("Set CODE_COMMIT to the verified full weak-label implementation commit.")
if ALLOW_WEAK_LABELS is not True:
    raise ValueError("This separate notebook requires explicit experimental weak-label opt-in.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
drive.mount("/content/drive")


Fetch exactly the selected code and install the separate learning runtime. If package versions change or cached modules are stale, this cell stops before ML imports: choose **Runtime → Restart session**, then **Run all** with the same configuration. Repeating cells in the same kernel cannot bypass the restart guard. ImageNet encoder weights are obtained only later through explicit training initialization.


In [ ]:
repository_path = Path("/content") / ("archaeotrace-code-" + CODE_COMMIT[:12])
if not repository_path.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY, str(repository_path)], check=True)
    subprocess.run(["git", "fetch", "origin", CODE_COMMIT], cwd=repository_path, check=True)
    subprocess.run(["git", "checkout", "--detach", CODE_COMMIT], cwd=repository_path, check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repository_path, text=True).strip()
if actual_commit != CODE_COMMIT:
    raise RuntimeError("Existing checkout does not match the selected commit.")
# Colab may already have imported Pillow/NumPy before package installation.
# Installing new files does not replace modules already cached in this kernel.
import hashlib, importlib, json, os, uuid
from importlib.metadata import PackageNotFoundError, version

runtime_distributions = (
    "torch", "torchvision", "segmentation-models-pytorch", "numpy", "Pillow", "onnx", "onnxruntime",
)

def _installed_learning_versions():
    versions = {}
    for distribution in runtime_distributions:
        try:
            versions[distribution] = version(distribution)
        except PackageNotFoundError:
            versions[distribution] = None
    return versions

# sys survives notebook-cell reruns, while a real kernel restart creates a new
# identity. Random lifetime identity avoids operating-system PID reuse.
if not hasattr(sys, "_archaeotrace_kernel_lifetime"):
    sys._archaeotrace_kernel_lifetime = uuid.uuid4().hex
kernel_lifetime = sys._archaeotrace_kernel_lifetime
requirements_path = repository_path / "requirements-learning.txt"
requirements_digest = hashlib.sha256(requirements_path.read_bytes()).hexdigest()
restart_marker = Path("/content") / (".archaeotrace-package-restart-" + requirements_digest[:16] + ".json")
restart_message = (
    "Learning packages changed or stale modules are loaded. In Colab choose "
    "Runtime > Restart session, then Run all with the same configuration. "
    "Do not continue individual training cells in this kernel."
)
if restart_marker.exists():
    marker = json.loads(restart_marker.read_text())
    if marker.get("kernel_lifetime") == kernel_lifetime:
        raise RuntimeError(restart_message)
versions_before = _installed_learning_versions()
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)], check=True)
importlib.invalidate_caches()
versions_after = _installed_learning_versions()
changed = [name for name in runtime_distributions if versions_before[name] != versions_after[name]]
loaded_mismatches = []
for module_name, distribution in (
    ("torch", "torch"), ("torchvision", "torchvision"),
    ("segmentation_models_pytorch", "segmentation-models-pytorch"),
    ("numpy", "numpy"), ("PIL", "Pillow"), ("onnx", "onnx"), ("onnxruntime", "onnxruntime"),
):
    module = sys.modules.get(module_name)
    loaded = getattr(module, "__version__", None) if module is not None else None
    installed = versions_after[distribution]
    if loaded is not None and installed is not None:
        if str(loaded).split("+", 1)[0] != str(installed).split("+", 1)[0]:
            loaded_mismatches.append(module_name)
if changed or loaded_mismatches:
    restart_marker.write_text(json.dumps({
        "kernel_lifetime": kernel_lifetime, "requirements_sha256": requirements_digest,
        "changed_distributions": changed, "stale_modules": loaded_mismatches,
    }, indent=2) + "\n")
    print("Restart required; changed:", changed, "stale loaded modules:", loaded_mismatches, flush=True)
    raise RuntimeError(restart_message)
restart_marker.unlink(missing_ok=True)
print("Learning packages verified in current kernel:", versions_after, flush=True)
sys.path.insert(0, str(repository_path))
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs an existing CUDA GPU runtime.")
print("Verified code:", actual_commit, "GPU:", torch.cuda.get_device_name(0))


Extract the frozen weak-label ZIP from Drive into the Colab local disk, rejecting unsafe archive paths and symbolic links. If the complete ZIP is absent, verify and atomically assemble ordered upload parts using `DATASET_PARTS_MANIFEST`; each part and the complete archive must match exact byte counts and SHA-256. Global split/freeze checks read manifest metadata; only the train/validation loaders verify and open learning image and label bytes. Checkpoints and exports stay on Drive. Keep this dataset and run directory separate from human-reviewed pilot results.


In [ ]:
import hashlib, json, os, re, shutil, stat, tempfile, zipfile
from pathlib import PurePosixPath

def _upload_sha256(path):
    result = hashlib.sha256()
    byte_count = 0
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(block)
            byte_count += len(block)
    return result.hexdigest(), byte_count


def assemble_upload_parts(parts_manifest_path, local_root):
    # Split uploads are a transport fallback, not a different training dataset.
    parts_manifest_path = Path(parts_manifest_path)
    if parts_manifest_path.is_symlink() or not parts_manifest_path.is_file():
        raise ValueError("Upload parts manifest must be a regular file.")
    declared_upload = json.loads(parts_manifest_path.read_text(encoding="utf-8"))
    if (not isinstance(declared_upload, dict)
        or declared_upload.get("schema_version") != "archaeotrace-colab-upload-parts/1"
        or declared_upload.get("archive_name") != "weak-dataset.zip"):
        raise ValueError("Unsupported split-upload manifest.")
    expected_hash = declared_upload.get("archive_sha256")
    expected_bytes = declared_upload.get("archive_bytes")
    parts = declared_upload.get("parts")
    if (not isinstance(expected_hash, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_hash)
        or type(expected_bytes) is not int or expected_bytes <= 0
        or not isinstance(parts, list) or not parts):
        raise ValueError("Upload archive needs an exact SHA-256, byte count and ordered parts.")
    names = set()
    part_paths = []
    for part in parts:
        if not isinstance(part, dict):
            raise ValueError("Each upload part must declare its name, SHA-256 and bytes.")
        name, digest, size = part.get("name"), part.get("sha256"), part.get("bytes")
        if (not isinstance(name, str) or not name or name in (".", "..")
            or "/" in name or "\\" in name or "\x00" in name
            or PurePosixPath(name).name != name or name in names
            or not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest)
            or type(size) is not int or size <= 0):
            raise ValueError("Upload part names must be unique basenames with valid SHA-256 and bytes.")
        names.add(name)
        part_path = parts_manifest_path.parent / name
        if part_path.is_symlink() or not part_path.is_file() or part_path.stat().st_size != size:
            raise ValueError("Upload part is missing, linked or has the wrong size: " + name)
        part_paths.append((part_path, digest, size))
    if sum(size for _path, _digest, size in part_paths) != expected_bytes:
        raise ValueError("Declared part sizes do not equal the complete archive size.")
    local_root = Path(local_root)
    local_root.mkdir(parents=True, exist_ok=True)
    assembled = local_root / ("archaeotrace-upload-" + expected_hash + ".zip")
    if assembled.is_symlink():
        raise ValueError("Local assembled archive cannot be a symbolic link.")
    descriptor, temporary = tempfile.mkstemp(prefix=".archaeotrace-upload-", dir=local_root)
    try:
        full_hash = hashlib.sha256()
        full_bytes = 0
        with os.fdopen(descriptor, "wb") as destination:
            for part_path, expected_part_hash, expected_part_bytes in part_paths:
                part_hash = hashlib.sha256()
                part_bytes = 0
                # Recheck immediately before reading each part.
                if part_path.is_symlink() or not part_path.is_file():
                    raise ValueError("Upload part changed to a missing or linked file.")
                with part_path.open("rb") as stream:
                    for block in iter(lambda: stream.read(1024 * 1024), b""):
                        destination.write(block)
                        part_hash.update(block)
                        full_hash.update(block)
                        part_bytes += len(block)
                        full_bytes += len(block)
                if part_bytes != expected_part_bytes or part_hash.hexdigest() != expected_part_hash:
                    raise ValueError("Upload part digest/size mismatch: " + part_path.name)
            destination.flush()
            os.fsync(destination.fileno())
        if full_bytes != expected_bytes or full_hash.hexdigest() != expected_hash:
            raise ValueError("Assembled upload archive digest/size mismatch.")
        os.replace(temporary, assembled)
    finally:
        Path(temporary).unlink(missing_ok=True)
    verified_hash, verified_bytes = _upload_sha256(assembled)
    if verified_hash != expected_hash or verified_bytes != expected_bytes:
        raise RuntimeError("Published assembled archive failed full re-verification.")
    print("Verified split upload:", len(parts), "parts;", verified_bytes,
          "bytes; SHA-256:", verified_hash, flush=True)
    return assembled


archive = Path(DATASET_ZIP)
if archive.is_symlink():
    raise ValueError("Dataset archive cannot be a symbolic link.")
if not archive.exists():
    archive = assemble_upload_parts(DATASET_PARTS_MANIFEST, Path("/content"))
if not archive.is_file():
    raise FileNotFoundError(archive)
archive_hasher = hashlib.sha256()
with archive.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        archive_hasher.update(block)
archive_hash = archive_hasher.hexdigest()
data_path = Path("/content") / ("archaeotrace-data-" + archive_hash[:16])
if not data_path.exists():
    stage = Path(tempfile.mkdtemp(prefix=".archaeotrace-data-", dir="/content"))
    try:
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                relative = PurePosixPath(member.filename)
                mode = member.external_attr >> 16
                if relative.is_absolute() or ".." in relative.parts or "\\" in member.filename or stat.S_ISLNK(mode):
                    raise ValueError("Unsafe archive member: " + member.filename)
                target = stage.joinpath(*relative.parts)
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with zipped.open(member) as source, target.open("xb") as destination:
                        shutil.copyfileobj(source, destination)
        (stage / ".archive-sha256").write_text(archive_hash)
        os.rename(stage, data_path)
    finally:
        if stage.exists():
            shutil.rmtree(stage)
if (data_path / ".archive-sha256").read_text() != archive_hash:
    raise RuntimeError("Local extraction does not match the selected archive.")
manifest_path = (data_path / MANIFEST_RELATIVE_PATH).resolve()
if not manifest_path.is_relative_to(data_path.resolve()):
    raise ValueError("Manifest must remain inside the extracted dataset.")
from learning.dataset import ContourDataset, load_dataset_manifest
from collections import Counter

# Global anti-leak/freeze checks use declared metadata, not protected labels.
frozen = load_dataset_manifest(manifest_path, verify_files=False)
if frozen.payload.get("dataset_id") == "usgs-contour-pilot-16x144-v1":
    raise ValueError("Weak-label data must have a separate dataset identity.")
experiment = frozen.payload.get("weak_label_experiment", {})
if (experiment.get("enabled") is not True
    or experiment.get("human_review_required") is not True
    or experiment.get("review_level") != "automated_weak_labels"):
    raise ValueError("Dataset must explicitly declare an unreviewed weak-label experiment.")
if frozen.payload.get("dataset_id") != DATASET_ID:
    raise ValueError("Unexpected weak-label dataset identity.")
sources = frozen.payload["sources"]
if dict(Counter(source["split"] for source in sources)) != EXPECTED_SOURCE_COUNTS:
    raise ValueError("Source cohort differs from the frozen sheet assignment.")
declared = frozen.payload["samples"]
if dict(Counter(sample["split"] for sample in declared)) != EXPECTED_SAMPLE_COUNTS:
    raise ValueError("Crop cohort differs from the frozen split counts.")
for sample in declared:
    annotation = sample.get("annotation", {})
    if annotation.get("status") != "machine_labeled":
        raise ValueError("Freeze all machine-label revisions before weak-label training.")
    if any(not re.fullmatch(r"[0-9a-f]{64}", annotation.get(key, "")) for key in ("sha256", "revision")):
        raise ValueError("Each weak annotation needs its exact revision and file digest.")
    if sample.get("prompt") or sample.get("reference_file") or sample.get("reference_sha256"):
        raise ValueError("AI weak labels cannot supply formal evaluation reference targets.")
training = ContourDataset(manifest_path, "train", allow_weak_labels=ALLOW_WEAK_LABELS)
validation = ContourDataset(manifest_path, "validation", allow_weak_labels=ALLOW_WEAK_LABELS)
print("Dataset SHA-256:", training.manifest_sha256)
print("Weak-label train/validation:", len(training), len(validation))
print("Protected calibration/holdout bytes remain unopened for training; human review remains required.")
run_root = Path(OUTPUT_DIRECTORY)
run_root.mkdir(parents=True, exist_ok=True)


Train or resume all three seeds on CUDA. The training loop reports epoch loss and actual optimizer progress, and stores the complete state on Drive every epoch or after five minutes at the next optimizer boundary. A CUDA OOM stops with the last valid checkpoint intact. Set `OOM_RETRY=True` and rerun this cell to explicitly resume at batch 1 / accumulation 16. Reconnection uses the same dataset, code, runtime versions and output directory. Revised labels require a new experiment directory.


In [ ]:
from learning.train import TrainConfig, load_checkpoint, train_run

results = {}
for seed in SEEDS:
    run_directory = run_root / ("seed-" + str(seed))
    latest = run_directory / "latest.pt"
    resume = latest if RESUME_EXISTING and latest.exists() else None
    config = TrainConfig(seed=seed, **TRAIN_OPTIONS)
    if resume:
        restored = load_checkpoint(resume)
        config = TrainConfig(**restored["config"])
        if config.seed != seed:
            raise ValueError("Resume checkpoint seed does not match its run directory.")
        if any(getattr(config, key) != value for key, value in TRAIN_OPTIONS.items()):
            raise ValueError("Resume checkpoint used different augmentation options.")
        if (restored.get("allow_weak_labels") is not True
            or restored.get("human_review_required") is not True
            or restored.get("review_level") != "automated_weak_labels"):
            raise ValueError("Resume checkpoint belongs to a different annotation-review scope.")
        del restored
    print("Starting weak-label seed", seed, "on CUDA", torch.cuda.get_device_name(0),
          "resume:", bool(resume), flush=True)
    results[seed] = train_run(
        manifest_path, run_directory, config=config, resume=resume,
        initialize_pretrained=resume is None, device="cuda",
        retry_oom=OOM_RETRY and resume is not None, allow_weak_labels=ALLOW_WEAK_LABELS,
    )
    print("Seed", seed, "stopped:", results[seed]["stopped_reason"],
          "epochs:", results[seed]["progress"]["epoch"],
          "optimizer steps:", results[seed]["progress"]["optimizer_steps"], flush=True)


Select the smallest weak-label validation loss, with ascending seed as the tie-break. Freeze `selected-model.json` before any protected sample access. Export only that seed's best checkpoint, with explicit weak-label provenance and CPU PyTorch/ONNX probability error at most 1e-4. This export is experimental and does not replace the human-reviewed Calibration/Holdout acceptance process. Fixed 512 tiles with 384 cores and 64 halos are pan-consistent and do not promise whole-image equivalence.


In [ ]:
import json, math
from learning.export import export_checkpoint, validate_validation_selection
from learning.train import canonical_sha256
from ai_vectorizer.core.learned_model import inspect_bundle

# Candidate ranking uses only frozen train/validation weak labels.
candidates = []
for seed in SEEDS:
    checkpoint = run_root / ("seed-" + str(seed)) / "best.pt"
    record = load_checkpoint(checkpoint)
    loss = record["progress"]["best_validation"]
    if loss is None or not math.isfinite(loss):
        raise ValueError("Each seed must have a finite validation-selected checkpoint.")
    if record["dataset_manifest_sha256"] != training.manifest_sha256 or record["code"]["commit"] != CODE_COMMIT:
        raise ValueError("Candidate dataset/code identity differs from the pinned run.")
    if (record.get("allow_weak_labels") is not True
        or record.get("human_review_required") is not True
        or record.get("review_level") != "automated_weak_labels"):
        raise ValueError("Each candidate must retain its unreviewed weak-label scope.")
    if record["synthetic_diagnostic"]:
        raise ValueError("Synthetic diagnostics cannot be selected as real pilot candidates.")
    candidates.append({
        "seed": seed, "validation_loss": loss, "checkpoint": str(checkpoint),
        "checkpoint_sha256": record["checkpoint_file_sha256"],
        "training_config_sha256": record["config_sha256"],
        "training_source_sha256": record["code"]["source_sha256"],
    })
    del record
selected = min(candidates, key=lambda value: (value["validation_loss"], value["seed"]))
selection = {
    "schema_version": "archaeotrace-validation-selection/1",
    "selection_uses": "validation_only", "tie_break": "ascending_seed",
    "dataset_manifest_sha256": training.manifest_sha256, "code_commit": CODE_COMMIT,
    "selected": selected, "candidates": candidates,
    "calibration_accessed": False, "holdout_accessed": False,
    "publication_ranking_eligible": False,
    "allow_weak_labels": True, "human_review_required": True,
    "review_level": "automated_weak_labels",
}
selection_path = run_root / "selected-model.json"
if selection_path.name != "selected-model.json" or selection_path.parent.resolve() != run_root.resolve():
    raise ValueError("Frozen selection must remain at the experiment's selected-model.json.")
if selection_path.is_symlink():
    raise ValueError("Frozen selection cannot be a symbolic link.")
if selection_path.exists():
    if not selection_path.is_file():
        raise ValueError("Frozen selection must be a regular file.")
    # An interrupted exclusive-create/rename may leave only a zero-byte claim.
    # Recover exactly that placeholder; valid or malformed nonempty records
    # remain frozen and are never overwritten by this recovery path.
    placeholder = selection_path.stat()
    if placeholder.st_size == 0 and selection_path.read_bytes() == b"":
        current = selection_path.stat()
        if (current.st_ino, current.st_size, current.st_mtime_ns) != (
            placeholder.st_ino, placeholder.st_size, placeholder.st_mtime_ns
        ):
            raise RuntimeError("Frozen selection changed while checking an empty placeholder.")
        selection_path.unlink()
        print("Recovered interrupted empty selected-model.json placeholder.", flush=True)
if selection_path.exists():
    if json.loads(selection_path.read_text()) != selection:
        raise RuntimeError("Candidate selection is already frozen; use a new experiment directory.")
else:
    descriptor, temporary = tempfile.mkstemp(prefix=".selection-", dir=run_root)
    try:
        with os.fdopen(descriptor, "w") as stream:
            json.dump(selection, stream, indent=2, allow_nan=False)
            stream.flush()
            os.fsync(stream.fileno())
        # Drive mounts support rename, while hard links may be unavailable.
        with selection_path.open("x"):
            pass
        os.replace(temporary, selection_path)
    finally:
        Path(temporary).unlink(missing_ok=True)
checkpoint = Path(selected["checkpoint"])
export_directory = checkpoint.parent / ("bundle-" + selected["checkpoint_sha256"][:12])
if export_directory.exists():
    existing = inspect_bundle(export_directory)
    if not existing.ready or existing.manifest["provenance"]["training_checkpoint_sha256"] != selected["checkpoint_sha256"]:
        raise RuntimeError("Existing bundle is unavailable or belongs to another checkpoint.")
    exported = existing.manifest
else:
    exported = export_checkpoint(checkpoint, export_directory, manifest_path,
        license_spdx=MODEL_LICENSE, selection_file=selection_path,
        model_id=MODEL_ID, allow_weak_labels=ALLOW_WEAK_LABELS)
if exported["provenance"]["training_checkpoint_sha256"] != selected["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint changed after validation selection.")
provenance = exported["provenance"]
if (provenance.get("allow_weak_labels") is not True
    or provenance.get("human_review_required") is not True
    or provenance.get("review_level") != "automated_weak_labels"
    or provenance.get("research_ready") is not False):
    raise RuntimeError("Exported bundle must retain unreviewed, experimental weak-label provenance.")
validate_validation_selection(selection, provenance)
if (provenance.get("validation_selection") != selection
    or provenance.get("validation_selection_sha256") != canonical_sha256(selection)
    or provenance.get("validation_selection_file_sha256") != hashlib.sha256(selection_path.read_bytes()).hexdigest()):
    raise RuntimeError("Exported bundle does not bind the frozen validation selection.")
print("Weak-label validation-only selected seed:", selected["seed"], "bundle:", export_directory)
print("Model SHA-256:", exported["model"]["sha256"], "frozen selection:", selection_path)
print("Experimental AI weak-label model: human review and independent contour evaluation remain required.")
